# Phase 1: WESAD Dataset Exploration & Inspection

This notebook performs exploratory analysis on the **WESAD (Wearable Stress and Affect Detection)** dataset.

## Objectives:
1. Detect and verify the WESAD folder structure and subject directories (`S2` through `S17`).
2. Inspect chest (700 Hz) and wrist (4-64 Hz) sensor modalities.
3. Check sampling frequencies, array shapes, and missing/NaN values.
4. Verify label distribution (Baseline, Stress, Amusement, Meditation, Transient).
5. Plot representative physiological waveforms comparing Baseline vs Stress states.

In [ ]:
import sys
from pathlib import Path
# Add root directory to sys.path
sys.path.append(str(Path.cwd().parent))

from src.physiological.preprocessing import load_subject_data, inspect_subject_signals, get_wesad_summary
import config
import numpy as np
import matplotlib.pyplot as plt

## 1. Dataset Scan & Label Distribution

In [ ]:
summary = get_wesad_summary()
print(f"Total Subjects Detected: {summary['num_subjects']}")
print(f"Subjects: {summary['subjects']}")
print("\nTotal Samples per Label (at 700Hz):")
for lbl, count in sorted(summary['total_label_counts'].items()):
    name = config.WESAD_LABEL_MAP.get(lbl, f'Other ({lbl})')
    mins = count / 700 / 60
    print(f"  Label {lbl} [{name:11s}]: {count:10,d} samples ({mins:6.2f} mins)")

## 2. Inspect Single Representative Subject (S2)

In [ ]:
s2_raw = load_subject_data('S2')
s2_info = inspect_subject_signals(s2_raw)
print('S2 Chest Signals:')
for k, v in s2_info['chest'].items():
    print(f"  {k:5s} -> Shape: {v['shape']}, NaN: {v['has_nan']}, Range: [{v['min']:.2f}, {v['max']:.2f}]")

print('\nS2 Wrist Signals:')
for k, v in s2_info['wrist'].items():
    print(f"  {k:5s} -> Freq: {v['sampling_hz']}Hz, Shape: {v['shape']}, NaN: {v['has_nan']}, Range: [{v['min']:.2f}, {v['max']:.2f}]")

## 3. Visualize Waveforms: Baseline vs Stress

In [ ]:
signals = s2_raw['signal']
labels = s2_raw['label']

# Select representative 10s windows
b_idx = np.where(labels == 1)[0][1000]
s_idx = np.where(labels == 2)[0][1000]
n_samples = 700 * 10
t = np.linspace(0, 10, n_samples)

fig, ax = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
ax[0, 0].plot(t, signals['chest']['ECG'][b_idx:b_idx+n_samples], color='blue')
ax[0, 0].set_title('Baseline ECG')
ax[0, 1].plot(t, signals['chest']['ECG'][s_idx:s_idx+n_samples], color='red')
ax[0, 1].set_title('Stress ECG')

ax[1, 0].plot(t, signals['chest']['EDA'][b_idx:b_idx+n_samples], color='green')
ax[1, 0].set_title('Baseline EDA')
ax[1, 1].plot(t, signals['chest']['EDA'][s_idx:s_idx+n_samples], color='orange')
ax[1, 1].set_title('Stress EDA')

ax[2, 0].plot(t, signals['chest']['Resp'][b_idx:b_idx+n_samples], color='purple')
ax[2, 0].set_title('Baseline Respiration')
ax[2, 1].plot(t, signals['chest']['Resp'][s_idx:s_idx+n_samples], color='brown')
ax[2, 1].set_title('Stress Respiration')

plt.tight_layout()
plt.show()